# 05 — LoRA fine-tuning (`peft` + TRL `SFTTrainer`)

Parameter-efficient fine-tuning: freeze the base weights and train small low-rank adapters
with **`peft`**, driven by **TRL**'s `SFTTrainer` over a raw-text corpus (continued-LM /
causal-LM loss over the whole sequence). The adapter is a few MB and is applied on top of
the base at inference — no merge needed.

**Library focus:** `peft.LoraConfig`, `trl.SFTConfig`, `trl.SFTTrainer`.

In [ ]:
import pathlib

MODEL = "Qwen/Qwen3.5-4B"
LIMIT = 256  # tiny pilot; raise to ~10k for a real run
OUT = pathlib.Path("/workspaces/llm-research/runs/_examples_05_lora")

## Data — a raw-text dataset with a `text` column

`SFTTrainer` tokenizes a named text column itself. Here we use the medical domain corpus.

In [ ]:
from datasets import load_dataset

ds = load_dataset("MedRAG/textbooks", split="train").select(range(LIMIT))
ds = ds.rename_column("content", "text").select_columns(["text"])
print(ds, "\n", ds[0]["text"][:160], "...")

## Configure LoRA + the trainer

`target_modules="all-linear"` lets peft attach adapters to every linear layer (architecture-
agnostic). `SFTConfig` is a `TrainingArguments` subclass; `packing=False` keeps samples
separate, `max_length` truncates. Set `report_to="wandb"` to log curves to Weights & Biases.

In [ ]:
from peft import LoraConfig

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

In [ ]:
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

cfg = SFTConfig(
    output_dir=str(OUT),
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    bf16=True,
    logging_steps=10,
    report_to="none",  # "wandb" to log to Weights & Biases
    seed=0,
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)

trainer = SFTTrainer(
    model=MODEL,
    args=cfg,
    train_dataset=ds,
    processing_class=tok,
    peft_config=lora,  # <- this makes it LoRA (vs. full fine-tuning, notebook 06)
)
trainable, total = trainer.model.get_nb_trainable_parameters()
print(f"trainable {trainable:,} / {total:,}  ({100 * trainable / total:.3f}%)")

In [ ]:
result = trainer.train()
adapter_dir = OUT / "adapter"
trainer.save_model(str(adapter_dir))  # saves the adapter + tokenizer (small)
print("train_loss:", result.training_loss, "| adapter ->", adapter_dir)

## Reload the adapter for inference

Load the base model, then attach the adapter with `PeftModel.from_pretrained`.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM

base = AutoModelForCausalLM.from_pretrained(MODEL, dtype="bfloat16", device_map="cuda")
tuned = PeftModel.from_pretrained(base, str(adapter_dir))
enc = tok("Myocardial infarction is", return_tensors="pt").to(tuned.device)
with torch.no_grad():
    out = tuned.generate(**enc, max_new_tokens=40, do_sample=False)
print(tok.decode(out[0], skip_special_tokens=True))

> **Project pointer:** `llm_core.training.sft.train_lora` wraps this (with dual-eval
> domain/general splits + early stopping for the forgetting study); `scripts/train.py`
> mixes corpora by weight (`configs/train/*.yaml`). Evaluate base+adapter on vLLM via
> `lora_local_path` (notebook 07) rather than merging a VLM base.